# Setting things up

## About this notebook

In this notebook, we embark on a journey to elevate the predictive capabilities of the California Housing Prices Dataset through advanced preprocessing using the ``ContinuousCarver`` pipeline. Renowned for its association-maximizing discretization, ``ContinuousCarver`` is a powerful Python tool designed to handle diverse data types—whether they be quantitative or qualitative. Our specific goal is to prepare the dataset for continuous regression tasks, such as predicting housing prices.

The California Housing Prices Dataset is a treasure trove of features, encompassing information on factors like square footage, bedrooms, location, and more. By employing ``ContinuousCarver``, we aim to seamlessly discretize both quantitative and qualitative features, tailoring them for optimal representation in our continuous regression models.

Throughout this notebook, we'll explore the intricacies of ``ContinuousCarver``'s discretization pipeline, witnessing its adaptability to a variety of data types. Whether it involves transforming square footage or encoding location information, ``ContinuousCarver`` ensures that each feature is finely tuned for our regression tasks.

Join us in this exploration as we leverage the power of ``ContinuousCarver`` to preprocess the California Housing Prices Dataset. Through effective feature engineering and discretization, our aim is to create a dataset that captures the nuanced relationships within the housing market, setting the stage for the development of accurate and impactful continuous regression models.

Let's dive in and uncover the potential of ``ContinuousCarver`` in transforming the California Housing Prices Dataset for optimal predictive modeling.


## Installation

In [1]:
# %pip install AutoCarver[jupyter]

## Califorinia Housing Prices Data

In this example notebook, we will use the California Housing Prices dataset.

The California Housing Prices dataset is a well-known dataset in the field of machine learning and statistics. It provides information about housing districts in California and is frequently used for regression analysis and predictive modeling tasks.

Comprising housing-related metrics for various districts in California, such as median house value, median income, housing median age, average rooms, average bedrooms, population, households, and more, the California Housing Prices dataset is a valuable resource for exploring the relationships between different features and predicting the median house values (continuous regression).

In [1]:
from sklearn import datasets

# Load dataset directly from sklearn
housing = datasets.fetch_california_housing(as_frame=True)

# conversion to pandas
housing_data = housing["data"]
housing_data[housing["target_names"][0]] = housing["target"]

# Display the first few rows of the dataset
housing_data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## Target type and Carver selection

In [2]:
target = "MedHouseVal"

housing_data[target].describe()

count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: MedHouseVal, dtype: float64

The target ``"MedHouseVal"`` is a continuous target of type ``float64`` used in a regression task. Hence we will use ``AutoCarver.ContinuousCarver`` and ``AutoCarver.selectors.RegressionSelector`` in following code blocks.

## Data Sampling

In [3]:
from sklearn.model_selection import train_test_split

# stratified sampling by target
train_set, dev_set = train_test_split(housing_data, test_size=0.33, random_state=42)

# checking target rate per dataset
train_set[target].mean(), dev_set[target].mean()

(np.float64(2.0666362048018514), np.float64(2.072459655020552))

## Picking up columns to Carve

In [4]:
train_set.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
5088,0.9809,19.0,3.187726,1.129964,726.0,2.620939,33.98,-118.28,1.214
17096,4.2232,33.0,6.189696,1.086651,1015.0,2.377049,37.46,-122.23,3.637
5617,3.5488,42.0,4.821577,1.095436,1044.0,4.331950,33.79,-118.26,2.056
20060,1.6469,24.0,4.274194,1.048387,1686.0,4.532258,35.87,-119.26,0.476
895,3.9909,14.0,4.608303,1.089350,2738.0,2.471119,37.54,-121.96,2.360


In [5]:
# column data types
train_set.dtypes

MedInc         float64
HouseAge       float64
AveRooms       float64
AveBedrms      float64
Population     float64
AveOccup       float64
Latitude       float64
Longitude      float64
MedHouseVal    float64
dtype: object

All features are quantitative continuous features at the exception of ``Latitude`` and ``Longitude`` which are geographical featues (not supported by ``AutoCarver`` as is). All other features will be added to the list of ``quantitative_features``.

In [6]:
from AutoCarver import Features

# lists of features per data type
features = Features(numericals=["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup"])

# Using AutoCarver

## AutoCarver settings

### Representativness of modalities

The attribute ``min_freq`` allows one to choose the minimum frequency per basic modalities. It is used:

- For quantitative features, to define the number of quantiles to initialy discretize the features with.

- For qualitative features, to define the threshold under which a modality is grouped to either a default value or its closest modality.

In [7]:
min_freq = 0.1

**Tip:** should be set between ``0.01`` (slower, preciser, less robust) and ``0.2`` (faster, more robust)

### Desired number of modalities

The attribute ``max_n_mod`` allows one to choose the maximum number of modalities per carved feature. It is used by **Carvers** has the upper limit of number of modalities per consecutive combination of modalities.

In [8]:
max_n_mod = 4

**Tip:** should be set between ``3`` (faster, more robust) and ``7`` (slower, preciser, less robust)

### Grouping NaNs

The attribute ``dropna`` allows one to choose whether or not ``nan`` should be grouped with another modality. If set to ``True``, **Carvers** will first find the most suitable combination of non-``nan`` values, and then test out all possible combinations with ``nan``.

In [9]:
dropna = False  # anyway, there are no nan in this dataset

#### Type of output carved features

The attribute ``ordinal_encoding`` allows one to choose the output type:

* Use ``True`` for integer output of ranked modalities (default)
* Use ``False`` for string output of modalities

In [10]:
ordinal_encoding = True

## Fitting AutoCarver

* First, all quantitative features are discretized:
    1. Using ``ContinuousDiscretizer`` for quantile discretization that keeps track of over-represented values (more frequent than ``min_freq``)
    2. Using ``OrdinalDiscretizer`` for any remaining under-represented values (less frequent than ``min_freq/2``) to be grouped with its closest modality

* Second, all features are carved following this recipe, for all classes of ``train_set[target]`` (except one):
    1. The raw distribution is printed out on provided ``train_set`` and ``dev_set``. It's the output of the discretization step
    2. Grouping modalities: all consecutive combinations of modalities are applied to ``train_set``
    3. Computing associations: the association metric (Krsuskal-Wallis' statistic, by default) is computed with the provided ``train_set[target]``
    4. Combinations are sorted in descending order by association value
    5. Testing robustness: finds the first combination that checks the following:
        - Representativness of modalities on ``train_set`` and ``dev_set`` (all should be more frequent than ``min_freq/2``)
        - Distinct target rates per consecutive modalities on ``train_set`` and ``dev_set`` 
        - No inversion of target rates between ``train_set`` and ``dev_set`` (same ordering of modalities by target rate)
    6. (Optional) If requested via ``dropna=True``, and if any, all combinations of modalities with ``nan`` are applied to ``train_set`` and steps 3. and 4. are run
    7. The carved distribution is printed out on provided ``train_set`` and ``dev_set``. It's the output of the carving step

In [11]:
from AutoCarver import ContinuousCarver
from AutoCarver.discretizers.utils.base_discretizer import ProcessingConfig

# intiating AutoCarver
auto_carver = ContinuousCarver(
    features=features,
    min_freq=min_freq,
    max_n_mod=max_n_mod,
    config=ProcessingConfig(dropna=dropna, ordinal_encoding=ordinal_encoding, verbose=True, copy=True),
)

# fitting on training sample, a dev sample can be specified to evaluate carving robustness
train_set_processed = auto_carver.fit_transform(train_set, train_set[target], X_dev=dev_set, y_dev=dev_set[target])

------
--- [QuantitativeDiscretizer] Fit Features(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup'])
 - [ContinuousDiscretizer] Fit Features(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup'])
 - [OrdinalDiscretizer] Fit Features(['HouseAge'])
------

---------
------ [ContinuousCarver] Fit Features(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup'])
--- [ContinuousCarver] Fit Quantitative('MedInc') (1/6)
 [ContinuousCarver] Raw distribution


,target_mean,frequency,count
x <= 1.60e+00,1.1102,0.0500,692
1.60e+00 < x <= 1.91e+00,1.1285,0.0500,691
1.91e+00 < x <= 2.15e+00,1.2198,0.0500,692
2.15e+00 < x <= 2.35e+00,1.3171,0.0500,691
2.35e+00 < x <= 2.57e+00,1.3817,0.0500,691
2.57e+00 < x <= 2.74e+00,1.5409,0.0500,692
2.74e+00 < x <= 2.98e+00,1.6159,0.0500,692
2.98e+00 < x <= 3.14e+00,1.6906,0.0499,690
3.14e+00 < x <= 3.32e+00,1.8232,0.0500,692
3.32e+00 < x <= 3.54e+00,1.9059,0.0500,691


 [ContinuousCarver] Carved distribution


X distribution 
 
 
   
 target_mean 
 frequency 
 count 
 
 
 
 
 x <= 2.57e+00 
 1.2314 
 0.2500 
 3457 
 
 
 2.57e+00 < x <= 3.97e+00 
 1.8016 
 0.3500 
 4840 
 
 
 3.97e+00 < x <= 5.54e+00 
 2.3587 
 0.2499 
 3456 
 
 
 5.54e+00 < x 
 3.5900 
 0.1501 
 2075 
 
 
 
 
 
 X_dev distribution 
 
 
 target_mean 
 frequency 
 count 
 
 
 
 
 1.2315 
 0.2554 
 1740 
 
 
 1.8222 
 0.3509 
 2390 
 
 
 2.3953 
 0.2446 
 1666 
 
 
 3.5721 
 0.1491 
 1016

--- [ContinuousCarver] Fit Quantitative('HouseAge') (2/6)
 [ContinuousCarver] Raw distribution


...

X distribution 
 
 
   
 target_mean 
 frequency 
 count 
 
 
 
 
 x <= 6.27e+02 
 2.0864 
 0.1503 
 2079 
 
 
 6.27e+02 < x <= 8.64e+02 
 2.1743 
 0.1498 
 2072 
 
 
 8.64e+02 < x <= 2.15e+03 
 2.0433 
 0.5498 
 7602 
 
 
 2.15e+03 < x 
 2.0250 
 0.1501 
 2075 
 
 
 
 
 
 X_dev distribution 
 
 
 target_mean 
 frequency 
 count 
 
 
 
 
 2.0867 
 0.1572 
 1071 
 
 
 2.1618 
 0.1536 
 1046 
 
 
 2.0607 
 0.5390 
 3672 
 
 
 2.0084 
 0.1502 
 1023

--- [ContinuousCarver] Fit Quantitative('AveOccup') (6/6)
 [ContinuousCarver] Raw distribution


,target_mean,frequency,count
x <= 1.870e+00,2.7122,0.0500,692
1.870e+00 < x <= 2.067e+00,2.6633,0.0500,691
2.067e+00 < x <= 2.225e+00,2.3373,0.0500,692
2.225e+00 < x <= 2.338e+00,2.3080,0.0500,691
2.338e+00 < x <= 2.432e+00,2.1976,0.0500,691
2.432e+00 < x <= 2.513e+00,2.2064,0.0500,692
2.513e+00 < x <= 2.595e+00,2.1736,0.0500,691
2.595e+00 < x <= 2.668e+00,2.1862,0.0500,691
2.668e+00 < x <= 2.743e+00,2.1378,0.0500,692
2.743e+00 < x <= 2.820e+00,2.1902,0.0500,691


 [ContinuousCarver] Carved distribution


X distribution 
 
 
   
 target_mean 
 frequency 
 count 
 
 
 
 
 x <= 2.22e+00 
 2.5709 
 0.1501 
 2075 
 
 
 2.22e+00 < x <= 3.07e+00 
 2.1681 
 0.5001 
 6915 
 
 
 3.07e+00 < x <= 3.61e+00 
 1.8729 
 0.1998 
 2763 
 
 
 3.61e+00 < x 
 1.4822 
 0.1501 
 2075 
 
 
 
 
 
 X_dev distribution 
 
 
 target_mean 
 frequency 
 count 
 
 
 
 
 2.5615 
 0.1461 
 995 
 
 
 2.1836 
 0.5129 
 3494 
 
 
 1.8527 
 0.1869 
 1273 
 
 
 1.5056 
 0.1541 
 1050

## AutoCarver analysis

### Carving Summary

In [20]:
auto_carver.summary

content  \
feature                    count  kruskal     n_mod label                               
Quantitative('MedInc')     3457.0 6037.182135 4     0                   x <= 2.57e+00   
                           4840.0 6037.182135 4     1        2.57e+00 < x <= 3.97e+00   
                           3456.0 6037.182135 4     2        3.97e+00 < x <= 5.54e+00   
                           2075.0 6037.182135 4     3                    5.54e+00 < x   
Quantitative('HouseAge')   4820.0 160.599610  4     0                   x <= 2.20e+01   
                           5734.0 160.599610  4     1        2.20e+01 < x <= 3.70e+01   
                           1791.0 160.599610  4     2        3.70e+01 < x <= 4.50e+01   
                           1483.0 160.599610  4     3                    4.50e+01 < x   
Quantitative('AveRooms')   8988.0 1401.052572 4     0                   x <= 5.69e+00   
                           2074.0 1401.052572 4     1        5.69e+00 < x <= 6.27e+00   
                           1383.0 1401.052572 4     2        6.27e+00 < x <= 6.95e+00   
                                                    3                    6.95e+00 < x   
Quantitative('AveBedrms')  7606.0 320.789845  4     0                  x <= 1.058e+00   
                           2765.0 320.789845  4     1      1.058e+00 < x <= 1.100e+00   
                           1382.0 320.789845  4     2      1.100e+00 < x <= 1.138e+00   
                           2075.0 320.789845  4     3                   1.138e+00 < x   
Quantitative('Population') 2079.0 16.109709   4     0                   x <= 6.27e+02   
                           2072.0 16.109709   4     1        6.27e+02 < x <= 8.64e+02   
                           7602.0 16.109709   4     2        8.64e+02 < x <= 2.15e+03   
                           2075.0 16.109709   4     3                    2.15e+03 < x   
Quantitative('AveOccup')   2075.0 991.408301  4     0                   x <= 2.22e+00   
                           6915.0 991.408301  4     1        2.22e+00 < x <= 3.07e+00   
                           2763.0 991.408301  4     2        3.07e+00 < x <= 3.61e+00   
                           2075.0 991.408301  4     3                    3.61e+00 < x   

                                                           target_mean  \
feature                    count  kruskal     n_mod label                
Quantitative('MedInc')     3457.0 6037.182135 4     0         1.231421   
                           4840.0 6037.182135 4     1         1.801562   
                           3456.0 6037.182135 4     2         2.358660   
                           2075.0 6037.182135 4     3         3.590040   
Quantitative('HouseAge')   4820.0 160.599610  4     0         1.949361   
                           5734.0 160.599610  4     1         2.062306   
                           1791.0 160.599610  4     2         2.055043   
                           1483.0 160.599610  4     3         2.478542   
Quantitative('AveRooms')   8988.0 1401.052572 4     0         1.821999   
                           2074.0 1401.052572 4     1         2.089595   
                           1383.0 1401.052572 4     2         2.546315   
                                                    3         3.142406   
Quantitative('AveBedrms')  7606.0 320.789845  4     0         2.152832   
                           2765.0 320.789845  4     1         2.087773   
                           1382.0 320.789845  4     2         1.967066   
                           2075.0 320.789845  4     3         1.788831   
Quantitative('Population') 2079.0 16.109709   4     0         2.086394   
                           2072.0 16.109709   4     1         2.174297   
                           7602.0 16.109709   4     2         2.043255   
                           2075.0 16.109709   4     3         2.024995   
Quantitative('AveOccup')   2075.0 991.408301  4     0         2.570888   
                           6915.0 991.408301  4     1        

* As requested with ``ordinal_encoding=True``, output labels are integers of modalities

* For quantitative feature ``Population``, the selected combination of modalities groups populations as follows:
    * label ``0``: lower or equal to 630 people (``content="x <= 6.3e+02"``)
    * label ``1``: greater than 630 people and lower or equal to 860 people  (``content="6.3e+02 < x <= 8.6e+02"``)
    * label ``2``: greater than 860 people and lower or equal to 2200 people (``content="8.6e+02 < x <= 2.2e+03"``)
    * label ``3``: higher than 2200 people (``content="2.2e+03 < x"``)

### Detailed overview of tested combinations

In [21]:
features["AveOccup"].history.head(7)

,info,kruskal,combination,n_mod,dropna,train,viable,dev
0,Raw distribution (n_mod=20>max_n_mod=4),1062.072498,"{'x <= 1.870e+00': 'x <= 1.870e+00', '1.870e+0...",20,False,NaN,NaN,NaN
1,Not viable,994.514410,"{'x <= 1.870e+00': 'x <= 1.870e+00', '1.870e+0...",4,False,"{'viable': True, 'info': ''}",False,"{'viable': False, 'info': 'Non-representative ..."
2,Not viable,994.504665,"{'x <= 1.870e+00': 'x <= 1.870e+00', '1.870e+0...",4,False,"{'viable': True, 'info': ''}",False,"{'viable': False, 'info': 'Non-representative ..."
3,Not viable,991.504255,"{'x <= 1.870e+00': 'x <= 1.870e+00', '1.870e+0...",4,False,"{'viable': True, 'info': ''}",False,"{'viable': False, 'info': 'Non-representative ..."
4,Best for kruskal and max_n_mod=4,991.408301,"{'x <= 1.870e+00': 'x <= 1.870e+00', '1.870e+0...",4,False,"{'viable': True, 'info': ''}",True,"{'viable': True, 'info': ''}"


In [22]:
features["AveOccup"].history.dev[1]

{'viable': False, 'info': 'Non-representative modality for min_freq=10.00%'}

* The most associated combination of feature ``AveOccup`` (the first tested out, where ``info!="Raw distribution"``) did not pass the viability tests. When looking in ``history.dev``:
    * ``"Non-representative modality for min_freq=10.00%"``: tells us that a modality is unstable between ``train_set`` and ``dev_set``

* For feature feature ``AveOccup``, the 4th combination is the first to pass tests:
    - ``viabe=True``
    - ``info="Best for kruskal and max_n_mod=4"``
    - Kruskal-Wallis' H with ``MedHouseVal`` is ``991.408301`` for this combination
    - Following combinations (less associated with the target) where not tested: ``info="Not checked"``

* For all combinations ``dropna=False`` means that it is not a combination in which ``nan``s are being grouped with other modalities (as requested with ``dropna=False``)

## Saving and Loading AutoCarver

### Saving

All **Carvers** can safely be stored as a .json file.

In [23]:
auto_carver.save("continuous_carver.json")

### Loading

**Carvers** can safely be loaded from a .json file.

In [24]:
from AutoCarver import ContinuousCarver

# loading json file
auto_carver = ContinuousCarver.load('continuous_carver.json')

## Applying AutoCarver

In [25]:
dev_set_processed = auto_carver.transform(dev_set)

In [27]:
dev_set_processed[auto_carver.features].apply(lambda u: u.value_counts(dropna=False, normalize=True))

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup
0.0,0.255432,0.340282,0.659278,0.558280,0.157223,0.146066
1.0,0.350851,0.414416,0.144157,0.200382,0.153553,0.512918
2.0,0.244568,0.138432,0.093511,0.095126,0.539049,0.186876
3.0,0.149149,0.106870,0.103053,0.146213,0.150176,0.154140


# Feature Selection
## Selectors settings
### Features to select from

Here all features have been carved using ``ContinuousCarver``, hence all features are qualitative.

### Number of features to select

The attribute ``n_best_per_type`` allows one to choose the number of features to be selected per data type (quantitative and qualitative).

In [28]:
n_best_per_type = 6 

## Using Selectors

In [29]:
from AutoCarver.selectors import RegressionSelector
from AutoCarver.discretizers.utils.base_discretizer import ProcessingConfig

# select the most target associated qualitative features
feature_selector = RegressionSelector(
    features=features,
    n_best_per_type=n_best_per_type,
    config=ProcessingConfig(verbose=True),  # displays statistics
)
best_features = feature_selector.fit(train_set_processed, train_set_processed[target]).selected_features
best_features

 [RegressionSelector] Selected Qualitative Features 


,feature,Nan,Mode,KruskalEtaSquaredMeasure,KruskalEtaSquaredRank,TschuprowtFilter,TschuprowtWith
0,Quantitative('MedInc'),0.0000,0.3500,0.4365,0.0000,0.0000,itself
2,Quantitative('AveRooms'),0.0000,0.6500,0.1011,1.0000,0.3854,MedInc
5,Quantitative('AveOccup'),0.0000,0.5001,0.0715,2.0000,0.1620,AveRooms
3,Quantitative('AveBedrms'),0.0000,0.5500,0.0230,3.0000,0.1395,MedInc
1,Quantitative('HouseAge'),0.0000,0.4147,0.0114,4.0000,0.1345,AveRooms
4,Quantitative('Population'),0.0000,0.5498,0.0009,5.0000,0.1464,HouseAge


Features(['MedInc', 'AveRooms', 'AveOccup', 'AveBedrms', 'HouseAge', 'Population'])

In [30]:
train_set_processed[best_features].head()

,MedInc,AveRooms,AveOccup,AveBedrms,HouseAge,Population
5088,0.0,0.0,1.0,2.0,0.0,1.0
17096,2.0,1.0,1.0,1.0,1.0,2.0
5617,1.0,0.0,3.0,1.0,2.0,2.0
20060,0.0,0.0,3.0,0.0,1.0,2.0
895,2.0,0.0,1.0,1.0,0.0,3.0


* Feature ``MedInc`` is the most associated with the target ``MedHouseVal``:
    - Kruskal-Wallis' H value is ``KruskalMeasure=6037.1821``
    - It has 0 % of NaNs (``NanMeasure=0.0000``) 
    - Its mode represents 35 % of observed data (``ModeMeasure=0.3500``)

* Feature ``AveRooms`` is strongly associated to feature ``MedInc``:
    - Tschuprow's T value is ``TschuprowtFilter=0.4015`` for ``TschuprowtWith=MedInc``

* Here, no feature where filtered out for there inter-feature association or over-represented values (no thresholds were set)

# Modeling
Fitting model on train data

In [31]:
from xgboost import XGBRegressor

model = XGBRegressor()
model.fit(train_set_processed[best_features], train_set_processed[target])

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

Saving model

In [24]:
model.save_model("regression_xgboost.json")

Prediction on dev dataset and performance

In [32]:
from sklearn.metrics import root_mean_squared_error

dev_pred = model.predict(dev_set_processed[best_features])
root_mean_squared_error(dev_set_processed[target], dev_pred)

0.7807707240408291

## What's next?

* Thanks to **Carvers** all of your features are now optimally processed for your regression task!
* As a final step towards your model, **Selectors** can prove to be handy tools to operate target optimal Data Pre-Selection, so make sure to check out [Selectors Examples](https://autocarver.readthedocs.io/en/latest/selectors_examples.html)!

## Well done!

Your commitment to achieving optimal results in continuous regression tasks shines through in your meticulous use of **AutoCarver**'s ``ContinuousCarver`` for data preprocessing. By fine-tuning and optimizing your dataset, you have set the stage for robust and accurate machine learning models.

The ``ContinuousCarver`` has proven to be a valuable ally in your pursuit of excellence, carving out a path toward enhanced feature representation and model interpretability. Your dedication to refining the data preprocessing steps reflects a commitment to extracting the maximum value from your datasets.

We extend our sincere appreciation for choosing **AutoCarver** as your companion in the data preprocessing journey. Your use of **AutoCarver** demonstrates a dedication to leveraging cutting-edge tools for achieving excellence in continuous regression tasks.

As you transition to the modeling phase, may the carefully crafted features and preprocessing steps contribute to the success of your predictive models. We're excited to see the impact of your work and are grateful for the opportunity to be part of your data science endeavors.

Thank you for trusting **AutoCarver**, and we wish you continued success in your data-driven ventures.